# ICPR 2026 experiment

Notebook for the experiment presented in the ICPR 2026 paper "Unifying Runtime Monitoring Approaches for Safety-Critical Machine Learning: Application to Vision-based Landing", M. Dario, F. Chenevier, K. Delmas, J. Guerin, J. Guiochet.

## Data download

<div class="alert alert-warning", style="text-align: justify">

This section allows to download the LARD (v1) dataset the public repo. Code can also be found in the dedicated notebook `data-download.ipynb` in the `./notebooks/` directory. Once the data is downloaded and unzipped, it should be available in a directory located at `PATH=../LARD_dataset` w.r.t. the project's root. Continue to next section to export the data to YOLO format.

</div>

*This section can be run independently from the sections below (needed libraries are re-imported)*

In [ ]:
# .
# ├── LARD_dataset/                   # Dataset directory (LARDv1)
# |   ├── LARD_test/                      # Folder once data unzip test
# |   ├── LARD_train/                     # Folder once data unzip train
# |   ├── LARD_test_real.zip              # Zip archive for test real 
# |   ├── LARD_test_synth.zip             # Zip archive for test synthetic
# |   ├── LARD_train_BIRK_LFST.zip        # ...
# |   ├── LARD_train_DAAG_DIAP.zip        
# |   └── ...  
# └── SwMF-LARD-YOLO-ICPR26/          # Project repo
#     ├── data/                           # YOLO-format exported datasets
#     ├── docs/
#     ├── notebooks/                      # Notebooks
#     ├── results/
#     └── swmf/                           # SwMF modules

### Setup

In [ ]:
import os
import subprocess
from pathlib import Path

In [ ]:
# Setup paths (EDIT)
DEFAULT_PATH_LARD_ARCHIVES = "../../LARD_dataset"  ## EDIT to path where zip files are saved
DEFAULT_PATH_LARD_DATASETS = "../../LARD_dataset"  ## KEEP AS IS (for compatibility with other notebooks)

In [ ]:
# List below the LARD archives to download from the web.
train_archives = [
    "LARD_train_BIRK_LFST.zip",
    "LARD_train_DAAG_DIAP.zip",
    "LARD_train_KMSY.zip",
    "LARD_train_LFMP_LFPO.zip",
    "LARD_train_LFQQ.zip",
    "LARD_train_LPPT_SRLI.zip",
    "LARD_train_VABB.zip",
    "LARD_train_domain_adaptation.zip",
]
tests_archives = [
    "LARD_test_real.zip",
    "LARD_test_synth.zip",
]

In [ ]:
# Set default paths and LARD url
lard_archives_dirpath = Path(DEFAULT_PATH_LARD_ARCHIVES).resolve()  # The lard archives path (before unzip)
lard_datasets_dirpath = Path(DEFAULT_PATH_LARD_DATASETS).resolve()  # The lard datasets path (after unzip)
url_dl = "https://share.deel.ai/s/H4iLKRmLkdBWqSt/download?path=%2Flard%2F1.0.0&files="
# url_dl = "https://share.deel.ai/s/3ZyWamJWrqzCf74?dir=/"

display(lard_archives_dirpath)
display(lard_datasets_dirpath)

### Download & Unzip

<div style="text-align: justify" class="alert alert-danger">

**Note**, the automated download using the link provided in the above cell does not work anymore. Please, **manually** download the LARD v1 datasets on the official data gouv repo [[link]](https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi:10.57745/MZSH2Y) (see instructions from the LARD GitHub [[link]](https://github.com/deel-ai/LARD/tree/LARD_V1)). Once downloaded, save the datasets following the provided structure at the beginning of the section, to ensure compatibility with previous automatic downloads.
</div>

In [ ]:
def dataset_download():
    """
    Function to call to download the LARD dataset
    """
    # Download datasets
    os.makedirs(lard_archives_dirpath.as_posix(), exist_ok=True)

    def _lard_download(src: str):
        print(f"Downloading ... {src}", flush=True)
        if not (lard_archives_dirpath / src).exists():
            subprocess.run(["wget", "-nc", url_dl + src, "-O", (lard_archives_dirpath / src).as_posix()], check=True)
            # %time !wget -nc {"\""+url_dl+src+"\""} -O {(lard_archives_dirpath / src).as_posix()}
        else:
            print(f"Target LARD archive already downloaded.", flush=True)

    for src in train_archives:
        _lard_download(src)

    for src in tests_archives:
        _lard_download(src)

In [ ]:
# Start download LARD (deprecated, manually download)
dataset_download()

In [ ]:
def dataset_unzip():
    """
    Function to call to unzip the archives
    """
    os.makedirs(lard_datasets_dirpath.as_posix(), exist_ok=True)

    def _lard_unzip(src: str, mod: str):
        print(f"Unzipping ... {src}", flush=True)
        if not (lard_datasets_dirpath / f"LARD_{mod}" / src.rpartition('.')[0]).exists():
            subprocess.run(["unzip", "-q", "-o", (lard_archives_dirpath / src).as_posix(), "-d", (lard_datasets_dirpath / f"LARD_{mod}").as_posix()], check=True)
            # %time !unzip -q -o ./{(lard_archives_dirpath / src).as_posix()} -d {(lard_datasets_dirpath / f'LARD_{mod}').as_posix()}
        else:
            tmp = (lard_datasets_dirpath / f"LARD_{mod}" / src.rpartition('.')[0])
            print(f"Target LARD dataset already exists. {tmp}", flush=True)

    for src in train_archives:
        _lard_unzip(src, "train")

    for src in tests_archives:
        _lard_unzip(src, "test")

In [ ]:
# Start unzip of archives
dataset_unzip()

## Data export

<div class="alert alert-warning", style="text-align: justify">

This section allows to export the LARDv1 dataset (previously downloaded) into a YOLO-format (see Ultralytics [doc](https://docs.ultralytics.com/datasets/detect#supported-dataset-formats) for more info). Once exported, the dataset should be available at `PATH=./data/datasets/name-of-the-exported-data/`. Code for export can also be found in the notebook `data-export.ipynb` in the `./notebooks/` directory.
</div>

*This section can be run independently from the sections below (needed libraries are re-imported)*

### Setup

In [2]:
# Add SwMF modules to PATH to import them
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
from pathlib import Path
from typing import (
    Union,
    Tuple,
    List,
)
import os
import shutil

import cv2
import json
import math
import numpy as np
import pandas as pd
import tqdm
import yaml

from swmf.monitors.ODD import comply_with_GLAC

Make sure to have the correct path to the LARD datasets; the same one used in previous section.

In [ ]:
LARD_DATASETS_PATH = "../../LARD_dataset"

In [ ]:
SHOULD_VERIFY_GLAC = True

# List of data archives to read and export
train_archives = [
    ("LARD_train_BIRK_LFST.zip", "LARD_train_BIRK_LFST.csv"),
    ("LARD_train_DAAG_DIAP.zip", "LARD_train_DAAG_DIAP.csv"),
    ("LARD_train_KMSY.zip", "LARD_train_KMSY.csv"),
    ("LARD_train_LFMP_LFPO.zip", "LARD_train_LFMP_LFPO.csv"),
    ("LARD_train_LFQQ.zip", "LARD_train_LFQQ.csv"),
    ("LARD_train_LPPT_SRLI.zip", "LARD_train_LPPT_SRLI.csv"),
    ("LARD_train_VABB.zip", "LARD_train_VABB.csv"),
]
tests_archives = [
    ("LARD_test_synth.zip", "LARD_test_synth.csv"),
]

LARD_dpath = Path(LARD_DATASETS_PATH).resolve()
LARD_train_dpath = LARD_dpath / 'LARD_train'
LARD_tests_dpath = LARD_dpath / 'LARD_test'

### Export (images, labels, metadata)

<div style="text-align: justify">

For each image, we export the following (meta)data:
- image (HxW pixels, RGB)
- label (cx, cy, h, w, class)
- metadata:
    - The airport name
    - The runway ID (= airport name + runway name)
    - The date (`YYYYMMDD`)
    - The time (`hh:mm:ss`)
    - ATD (*along track distance*)
    - VPA (*vertical path angle*)
    - LPA (*lateral path angle*)
    - $\phi$ (*roll angle*)
    - $\theta$ (*pitch angle*)
    - $\psi$ (*yaw angle*)

The metadata are supposed to be available at runtime, coming from
- external metadata (time-of-day, airport ID, runway ID)
- *exteroreceptive* sensors (position = ATD, VPA, LAP),
- *introspective* sensors (attitude = $\phi, \theta, \psi $),

</div>

In [ ]:
def convert_xyxy_to_xywh(
        bbox_xs: Union[np.ndarray, list],
        bbox_ys: Union[np.ndarray, list],
        img_w: int,
        img_h: int,
) -> np.ndarray: 
    """
    Convert the bounding box form xyxy (LARD format) to xywh (YOLO format).

    Note:
        The xyxy LARD bbox is in pixels while the xywh YOLO bbox is normalized to the image shape.

    Args:
        bbox_xs (Union[np.ndarray, list]): an array or list containing x's positions of the 4 bbox vertices.
        bbox_ys (Union[np.ndarray, list]): an array or list containing y's positions of the 4 bbox vertices.
        img_w (int): the original image's width
        img_h (int): the original image's height

    Returns:
        (np.ndarray) [4,] the YOLO-formated bbox.
    """
    xs = np.clip(bbox_xs, 0., img_w) / img_w  # Clip and normalize the box X coordinates
    ys = np.clip(bbox_ys, 0., img_h) / img_h  # Clip and normalize the box Y coordinates

    x_min = float(xs.min())
    x_max = float(xs.max())
    y_min = float(ys.min())
    y_max = float(ys.max())

    w  = x_max - x_min
    h  = y_max - y_min
    cx = x_min + w / 2.
    cy = y_min + h / 2.

    bbox = np.array([cx, cy, w, h])
    return bbox

In [ ]:
### HELPER ###
def get_runway_num(rn: str):
    """
    Util function to convert runway number into bearing angle
    """
    if not (isinstance(rn, int) or isinstance(rn, str)):
        raise TypeError("Input 'rn' must be either int or str.")
    
    if isinstance(rn, int):
        return rn
    else:
        s_type = type(rn)
        return int(s_type().join(filter(s_type.isdigit, rn)))
##############


### TEMP HOTFIX TO PROPERLY HANDLE YAW ANGLE ###
import pyproj
from skspatial.objects import Points

PATH_TO_RUNWAY_DATABASE = "../data/runways_database_all.json"


def ecef2llh(x, y, z):
    """
    https://github.com/deel-ai/LARD/blob/LARD_V1/src/ges/geo_utils.py
    """
    R = 6371010
    t = math.sqrt((x**2 + y**2 + z**2))

    if x == 0:
        if y < 0:
            lon = -math.pi / 2
        else:
            lon = math.pi / 2
    elif x < 0:
        if y < 0:
            lon = -math.pi + math.atan(y/x)
        else:
            lon = math.pi + math.atan(y/x)
    else:
        lon = math.atan(y/x)

    lat = math.asin(z/t)
    height = t - R
    return math.degrees(lat), math.degrees(lon), height 


def find_center(points):
    return Points(points).mean_center(return_centroid=True)


def find_azimuth_between_points(lon1, lat1, lon2, lat2):
    """
    https://github.com/deel-ai/LARD/blob/LARD_V1/src/ges/geo_utils.py#L258
    """
    return pyproj.Geod(ellps="WGS84").inv(lon1, lat1, lon2, lat2)


def get_runway_pts(database_file, airport_id, runway_id):
    """
    https://github.com/deel-ai/LARD/blob/LARD_V1/src/ges/ges_dataset.py#L86
    """
    with open(database_file, 'r') as f:
        runway_db = json.load(f)

    # Try adding 0 to the id...
    try:
        runway_pt = runway_db[airport_id][runway_id]
    except:
        runway_pt = runway_db[airport_id][f"{runway_id:0>2}"]

    _, ltp  = find_center([list(runway_pt['C']['position'].values()), list(runway_pt['D']['position'].values())])
    _, fpap = find_center([list(runway_pt['A']['position'].values()), list(runway_pt['B']['position'].values())])

    return runway_pt, ltp, fpap


def get_runway_yaw(database_file, airport_id, runway_id):
    """
    https://github.com/deel-ai/LARD/blob/LARD_V1/src/ges/ges_dataset.py#L107
    """
    _, ltp, fpap = get_runway_pts(database_file, airport_id, runway_id)

    fpap_lat, fpap_lon, _ = ecef2llh(*fpap)
    ltp_lat, ltp_lon, _ = ecef2llh(*ltp)
    rwy_psi = find_azimuth_between_points(ltp_lon, ltp_lat, fpap_lon, fpap_lat)
    
    print(airport_id, runway_id, '  \t', rwy_psi)
    return rwy_psi


def get_dist_between_angles(angle1, angle2):
    """
    Return the distance between two angles modulo 360
    """
    return min((angle1-angle2)%360, (angle2-angle1)%360)


def choose_runway_yaw(rwy_num, rwy_yaw_fwd, rwy_yaw_bwd):
    """
    Select the runway azimuth as close as the runway number as possible
    """
    if get_dist_between_angles(rwy_num*10, rwy_yaw_fwd) < get_dist_between_angles(rwy_num*10, rwy_yaw_bwd):
        return rwy_yaw_fwd
    else:
        return rwy_yaw_bwd

#####################################################

In [ ]:
def export_one_sample(
        img_sample: Tuple[int, pd.Series],
        dst_dpath: Path,
        new_shape: Tuple[int, int],
        tasks: List[str],
) -> None:
    """
    Export one sample (image + label + metadata).

    Args:
        img_sample (Tuple[int, pd.Series]): the sample index + info.
        dst_dpath (Path): the directory path to save the sample into.
        new_shape (Tuple[int, int]): the new shape of exported sample.
        tasks (List[str]): the list of ML tasks to export the sample for.
    """
    im_indx, im_info = img_sample

    im_fpath = im_info["image_dpath"] / im_info["image"].replace('\\', '/')
    im = np.array(cv2.cvtColor(cv2.imread(im_fpath), cv2.COLOR_BGR2RGB))  # [H, W, C]
    h = im.shape[0]
    w = im.shape[1]

    # Crop watermark if necessary (top and bottom)
    watermark = im_info["watermark_height"]
    if not math.isnan(watermark):
        watermark = int(watermark)
        im = im[watermark: -watermark, :, :]

    # Size up the image and save it
    new_img_fpath = dst_dpath / 'images' / im_info['split'] / f"{im_indx:06d}.jpg"
    if not new_img_fpath.exists():
        im = cv2.resize(im, new_shape, interpolation=cv2.INTER_NEAREST)
        os.makedirs(new_img_fpath.parent, exist_ok=True)
        cv2.imwrite(new_img_fpath, cv2.cvtColor(im, cv2.COLOR_RGB2BGR))

    # Save the metadata (independent of the ML task)
    new_met_fpath = dst_dpath / "metadatas" / im_info['split'] / f"{new_img_fpath.stem}.txt"
    if not new_met_fpath.exists():
        os.makedirs(new_met_fpath.parent, exist_ok=True)
        with open(new_met_fpath, "w") as f:
            f.write(";".join(im_info.loc[['airport','rwy_id','date','time','ATD','VPA','LPA','phi','theta','psi']].astype(str).to_list()))

    # Compute labels
    x = np.array([im_info[f"x_{k}"] for k in "ABCD"], dtype=np.float32)
    y = np.array([im_info[f"y_{k}"] for k in "ABCD"], dtype=np.float32)

    if not math.isnan(watermark):
        y -= watermark
        h -= watermark * 2
    bbox = convert_xyxy_to_xywh(x, y, w, h)

    # For object detection
    if "detect" in tasks:
        tmp_image_fpath = dst_dpath / "task_detect" / "images" / im_info['split'] / new_img_fpath.name
        tmp_label_fpath = dst_dpath / "task_detect" / "labels" / im_info['split'] / f"{new_img_fpath.stem}.txt"
        tmp_mdata_fpath = dst_dpath / "task_detect" / "metadatas" / im_info['split'] / f"{new_img_fpath.stem}.txt"

        if not tmp_label_fpath.exists():
            os.makedirs(tmp_image_fpath.parent, exist_ok=True)
            os.makedirs(tmp_label_fpath.parent, exist_ok=True)
            os.makedirs(tmp_mdata_fpath.parent, exist_ok=True)
            # Handle image
            os.symlink(new_img_fpath, tmp_image_fpath, target_is_directory=False)
            # Handle metadata    
            os.symlink(new_met_fpath, tmp_mdata_fpath, target_is_directory=False)
            # Handle label
            with open(tmp_label_fpath, "w") as f:
                f.write("%g %.6f %.6f %.6f %.6f\n" % (0, *bbox))

    # For images segmentation
    if "segment" in tasks:
        kpts = np.stack((x/w, y/h), axis=-1).reshape(-1).tolist()  # [x1,x2] & [y1,y2] => [x1,y1,x2,y2]
        tmp_image_fpath = dst_dpath / "task_segment" / "images" / im_info['split'] / new_img_fpath.name
        tmp_label_fpath = dst_dpath / "task_segment" / "labels" / im_info['split'] / f"{new_img_fpath.stem}.txt"
        tmp_mdata_fpath = dst_dpath / "task_segment" / "metadatas" / im_info['split'] / f"{new_img_fpath.stem}.txt"

        if not tmp_label_fpath.exists():
            os.makedirs(tmp_image_fpath.parent, exist_ok=True)
            os.makedirs(tmp_label_fpath.parent, exist_ok=True)
            os.makedirs(tmp_mdata_fpath.parent, exist_ok=True)
            # Handle images symlink
            os.symlink(new_img_fpath, tmp_image_fpath, target_is_directory=False)
            # Handle metadata
            os.symlink(new_met_fpath, tmp_mdata_fpath, target_is_directory=False)
            # Handle labels
            with open(tmp_label_fpath, "w") as f:
                f.write("0 " + " ".join([f'{p:.6f}' for p in kpts]) + "\n")


def export_one_split_set(
        archives: List[Tuple[str, str]],
        dataset_split: str,
        src_dpath: Path,
        dst_dpath: Path,
        new_image_shape: Tuple[int, int],
        tasks: List[str],
) -> None:
    """
    Export each sample of the given dataset split (train or test).

    Args:
        archives
        dataset_name
        src_dpath
        dst_dpath
        new_image_shape
        tasks
    """
    # Get the csv filepath from unzipped archives
    csv_fpaths = [src_dpath / zip_fname.rpartition('.')[0] / csv_fname for zip_fname, csv_fname in archives]

    # Get the data from csv files
    dfs = []
    for csv_fpath in csv_fpaths:
        dfi = pd.read_csv(csv_fpath, delimiter=";")
        dfi["image_dpath"] = csv_fpath.parent
        dfi["split"] = dataset_split
        dfs.append(dfi)
    df = pd.concat(dfs).reset_index(drop=True).reset_index(drop=False)
    df['index'] = df['index'].map(lambda x: f"{x:06d}")

    #######################################################
    ### Preproc the dataFrames (TODO: put elsewhere...) ###    
    df.rename(columns={'time': 'datetime'}, inplace=True)
    df['watermark_height'] = df['watermark_height'].fillna(0.0)
    df['airport'] = df['airport'].astype(str)
    df['runway'] = df['runway'].astype(str)
    df['rwy_id'] = df['airport'] + '|' + df['runway'].map(lambda x: f"{x:0>3}")
    df['date'] = df['datetime'].map(lambda x: x.split(' ')[0])
    df['time'] = df['datetime'].map(lambda x: x.split(' ')[1])
    df['ATD'] = df['along_track_distance'] * 1852            # Convert from NM to m
    df['LPA'] = np.deg2rad(df['lateral_path_angle'])         # Convert from degrees to radians
    df['VPA'] = np.deg2rad(df['vertical_path_angle']) *(-1)  # Convert from degrees to radians and opposite
    df['slant_distance'] *= 1.852   # Convert from NM to km
    df['phi'] = np.deg2rad(df['roll'])

    ## Compute the runway azimuth ##
    df['rwy_yaw'] = 0.0
    for airport_id in df['airport'].unique():
        for runway_id in df[df['airport']==airport_id]['runway'].unique():
            rwy_num = get_runway_num(runway_id)
            rwy_yaw = get_runway_yaw(PATH_TO_RUNWAY_DATABASE, airport_id, runway_id)
            df.loc[(df['airport']==airport_id)&(df['runway']==runway_id), 'rwy_yaw'] = choose_runway_yaw(
                rwy_num, rwy_yaw[0], rwy_yaw[1]
            )

    df['psi'] = np.deg2rad((df['yaw'] - df['rwy_yaw'] + 180) % 360 - 180)
    df['theta'] = np.deg2rad(df['pitch'] - 90)
    df['X'] = -df['ATD']
    df['Y'] = +df['ATD'] * np.tan(df['LPA'])
    df['Z'] = +df['ATD'] * np.tan(df['VPA'])
    #######################################################
    #######################################################


    #######################################################
    #######################################################
    if SHOULD_VERIFY_GLAC and dataset_split=="train":
        samples_ODD_ok = comply_with_GLAC(df).astype(bool)
        df = df[samples_ODD_ok]
        df = df.drop(columns=['index'])
        df = df.reset_index(drop=True).reset_index(drop=False)
        df['index'] = df['index'].map(lambda x: f"{x:06d}")
    #######################################################
    #######################################################


    # Save csv file of exported metadata {img -> idx}
    os.makedirs(dst_dpath / "images", exist_ok=True)
    df.to_csv(dst_dpath / "images" / f"{dataset_split}_metadata.csv", sep=';', index=False, columns=["index", "image", "airport", "rwy_id"], header=False)

    # Export each sample individually
    for s in tqdm.tqdm(df.iterrows()):
        export_one_sample(s, dst_dpath, new_image_shape, tasks)


def launch_export(
        dataset_path: Path,
        dataset_name: str,
        imgsz: Tuple[int, int],
        tasks: List[str] = ["detect"],
        override_data: bool = False,
) -> Path:
    """
    Launch the export of the datasets. Give a name to it, new image shape et specific to ML task.

    Args:
        dataset_path (Path): the destination path to save the exported datasets.
        dataset_name (str): the name given to the exported datasets.
        imgsz (Tuple[int, int]): the image resolution to export.
        tasks (List[str]): the ML tasks for which to export the datasets.
        override_data (bool): flag to allow for data override in destination path.

    Returns:
        (Path) the path to exported datasets.
    """
    dataset_full_path = dataset_path / dataset_name

    def _make_yaml_file():
        for task in tasks:
            d = {
                'path': (dataset_full_path / f"task_{task}").as_posix(),
                'train': "images/train",
                'valid': "",
                'test': "images/test",
                'nc': 1,
                'names': {0: "runway"},
            }
            with open((dataset_full_path / f"task_{task}" / "data.yaml").as_posix(), "w") as f:
                yaml.dump(d, f, sort_keys=False)

    if dataset_full_path.exists():
        if not override_data:
            print(f"Destination path already exists ({dataset_full_path.as_posix()}). No action done.")
            return dataset_full_path
        else:
            shutil.rmtree(dataset_full_path)
    
    os.makedirs(dataset_full_path, exist_ok=True)

    print("Exporting train set.", flush=True, end=" ")
    export_one_split_set(train_archives, "train", LARD_train_dpath, dataset_full_path, imgsz, tasks)
    print("Done.", flush=True)

    print("Exporting test set.", flush=True, end=" ")
    export_one_split_set(tests_archives, "test" , LARD_tests_dpath, dataset_full_path, imgsz, tasks)
    print("Done.", flush=True)

    print("Creating dataset YAML file.", flush=True, end=" ")
    _make_yaml_file()
    print("Done.")

    return dataset_full_path

Launch export of the LARD dataset to YOLO-format (about 20 minutes).

In [ ]:
# Initial dataset image 
LARD_EXPORT_RESOLUTION = (512, 512)  # W, H
LARD_EXPORT_PATH = "../data/datasets"
LARD_EXPORT_NAME = "lard_512x512_ICPR2026"    # Usually "lard_WxH"

lard_export_path = Path(LARD_EXPORT_PATH).resolve()

In [ ]:
lard_path = launch_export(
    lard_export_path,
    LARD_EXPORT_NAME,
    imgsz=LARD_EXPORT_RESOLUTION,
    tasks=["detect"],
    override_data=True,
)

print(f"Path to exported LARD: {lard_path.as_posix()}")

### Create train/valid split per runway

Make sure to run the entire section when (re)-generating the split, in order to always find the same images in the train and valid splits.

In [ ]:
SEED = 42
np.random.seed(42)

In [ ]:
metadata_df2 = pd.read_csv(lard_path / "images" / "train_metadata.csv", sep=";", header=None, dtype=str, names=['index', 'image', 'airport', 'rwy_id'])

trainval_runways = np.random.permutation(metadata_df2['rwy_id'].unique())
train_runways = trainval_runways[5:]  # Keep all but first 5 runways for training
valid_runways = trainval_runways[:5]  # Keep first 5 runways for validation

metadata_df2['split'] = np.where(metadata_df2['rwy_id'].isin(train_runways), "train", "valid")
metadata_df2.to_csv(lard_path / "split_trainval_per_runway.csv", columns=['index','split'], sep=';', index=False)

print("Selected VALID runways:")
print(valid_runways)
print(metadata_df2['split'].value_counts())

## YOLOv5 model

<div class="alert alert-warning", style="text-align: justify">

In this section we load an already trained YOLOv5n model, trained on the LARDv1 dataset, exported to YOLO-format, in 512x512 resolution. The model was trained for 20 epochs (it is not much, but YOLO model was quite good at performing runway detection, as shows train set evaluation). The code for YOLOv5 training can be found in the `ml-yolo.ipynb` notebook in the `./notebooks/` directory, the weights of the YOLOv5 model can be found in `./data/models/detect/yolov5n/yolo5n_20_epochs.pt`.
</div>

### Setup

In [ ]:
%load_ext skipkernel_extension

In [ ]:
should_skip_training_eval = False

In [ ]:
from pathlib import Path

import json
import numpy as np
import torch
import tqdm

# For YOLO models
from ultralytics import YOLO

# From SwMF package
from swmf.data import create_dataloader
from swmf.data import default_img_transform, default_lab_transform

from swmf.models.yolo_utils import DEFAULT_YOLO_IOU_THRESHOLD
from swmf.models.yolo_utils import DEFAULT_YOLO_OBJ_THRESHOLD
from swmf.models.yolo_utils import postproc_yolo_outputs, postproc_yolo_targets

from swmf.metrics import compute_metrics

In [ ]:
### Reproducibility ###
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
#######################

In [ ]:
PATH_LARD = "data/datasets/lard_512x512_ICPR2026"
PATH_YOLO = "data/models/detect/yolov5n_LARD_20epochs.pt"

In [ ]:
path_to_yolo = Path(PATH_YOLO).resolve()
assert path_to_yolo.exists(), f"ERROR: Specified path does not exist {path_to_yolo}"

path_to_lard = Path(PATH_LARD).resolve()
assert path_to_lard.exists(), f"ERROR: Specified path does not exist {path_to_lard}"

split_fpath = path_to_lard / "split_trainval_per_runway.csv"
split_dname = split_fpath.stem
assert split_fpath.exists(), f"ERROR: Specified path does not exist {split_fpath}"

In [ ]:
path_to_results = Path("results").resolve()
path_to_results = path_to_results / PATH_LARD.split('/')[-1] / "detect" / "yolov5n" / (path_to_lard.stem + "_" + split_fpath.stem) / "020_epochs" / split_dname
path_to_results

In [ ]:
### HYPERPARAMETERS ###
IOU_THRESHOLD = 0.70  # The IOU threshold to evaluate on
#######################

### Load LARD datasets

In [ ]:
IMGSZ = (512, 512)
BATCH = 8

trainval_images_path = path_to_lard / "task_detect" / "images" / "train"
trainval_labels_path = path_to_lard / "task_detect" / "labels" / "train"
trainval_loader = create_dataloader(
    image_dpath=trainval_images_path,
    label_dpath=trainval_labels_path,
    image_transform=default_img_transform(IMGSZ),
    label_transform=default_lab_transform(IMGSZ),
    yolo_task="detect",
    data_mode="train",
    batch_size=BATCH*4,
    shuffle=False,
)

train_loader = create_dataloader(
    dataset_dpath=path_to_lard,
    image_transform=default_img_transform(IMGSZ),
    label_transform=default_lab_transform(IMGSZ),
    yolo_task="detect",
    data_mode="train",
    split_fpath=split_fpath,
    split="train",
    using_metadata=False,
    batch_size=BATCH*4,
    shuffle=False,
)

valid_loader = create_dataloader(
    dataset_dpath=path_to_lard,
    image_transform=default_img_transform(IMGSZ),
    label_transform=default_lab_transform(IMGSZ),
    yolo_task="detect",
    data_mode="train",
    split_fpath=split_fpath,
    split="valid",
    using_metadata=False,
    batch_size=BATCH*4,
    shuffle=False,
)

test_loader = create_dataloader(
    dataset_dpath=path_to_lard,
    image_transform=default_img_transform(IMGSZ),
    label_transform=default_lab_transform(IMGSZ),
    yolo_task="detect",
    data_mode="test",
    using_metadata=True,
    batch_size=BATCH,
    shuffle=False,
)

### Load YOLO model

In [ ]:
yolo_model = YOLO(path_to_yolo)
yolo_model.fuse()
yolo_model.eval();

In [ ]:
ML_KWARGS = {
    'conf': DEFAULT_YOLO_OBJ_THRESHOLD,
    'iou' : DEFAULT_YOLO_IOU_THRESHOLD,
}

### Evaluate YOLO model on `valid` split

This section allows to calibrate the YOLOv5n confidence threshold with respect to the `valid` split performance.

In [ ]:
y_pred_valid = torch.Tensor([], device='cpu')
y_true_valid = torch.Tensor([], device='cpu')

tmp_imgs_n = 0
for X, y in tqdm.tqdm(valid_loader, desc="[VALID] Predicting..."):
    r = yolo_model.predict(X, verbose=False, **ML_KWARGS)

    y_pred_valid = torch.cat([y_pred_valid, postproc_yolo_outputs(r, imgs_n=tmp_imgs_n)])
    y_true_valid = torch.cat([y_true_valid, postproc_yolo_targets(y, imgs_n=tmp_imgs_n)])
    tmp_imgs_n += len(y)

# Compute and save the optimal confidence threshold
metrics_ml_valid = compute_metrics(y_pred=y_pred_valid, y_true=y_true_valid, iou_thresh=IOU_THRESHOLD)

print()
print("Metrics for YOLO [VALID]", f"(IOU={IOU_THRESHOLD})")
print("========================")
print("\n".join(f"{k} = {v}" for k, v in metrics_ml_valid.items()))
print()

# Optimal t_conf. Shape (n_iou_thresholds)
optimal_conf_threshold = metrics_ml_valid['c']
print("Optimal t_conf =", optimal_conf_threshold)

# Save results
result_path = path_to_results / "metrics" / "valid" / f"IOU_{int(IOU_THRESHOLD*100):03d}"
result_path.mkdir(exist_ok=True, parents=True)
with open(result_path /  f"yolo.json", "w") as f:
    json.dump(metrics_ml_valid, f)

### Evaluate YOLO model on `train` split

This section simply allows to evaluate the YOLO model on the `train` split for comparison.

In [ ]:
%%skip $should_skip_training_eval

y_pred_train = torch.Tensor([], device='cpu')
y_true_train = torch.Tensor([], device='cpu')

tmp_imgs_n = 0
for X, y in tqdm.tqdm(train_loader, desc="[TRAIN] Predicting..."):
    r = yolo_model.predict(X, verbose=False, **ML_KWARGS)

    y_pred_train = torch.cat([y_pred_train, postproc_yolo_outputs(r, imgs_n=tmp_imgs_n)])
    y_true_train = torch.cat([y_true_train, postproc_yolo_targets(y, imgs_n=tmp_imgs_n)])
    tmp_imgs_n += len(y)

# Compute and save the optimal confidence threshold
metrics_ml_train = compute_metrics(y_pred=y_pred_train, y_true=y_true_train, iou_thresh=IOU_THRESHOLD, t_conf=optimal_conf_threshold)

print()
print("Metrics for YOLO [TRAIN]", f"(IOU={IOU_THRESHOLD})")
print("========================")
print("\n".join(f"{k} = {v}" for k, v in metrics_ml_train.items()))
print()

# Save results
result_path = path_to_results / "metrics" / "train" / f"IOU_{int(IOU_THRESHOLD*100):03d}"
result_path.mkdir(exist_ok=True, parents=True)
with open(result_path / f"yolo.json", "w") as f:
    json.dump(metrics_ml_train, f)

## Runtime Monitoring with SwMF

### Setup

In [ ]:
import json
import numpy as np
import torch
import tqdm

# For monitors evaluation
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

from swmf.metrics import match_predictions
from swmf.metrics import compute_metrics, compute_safety_metrics

from swmf.monitors.ODD import ODD_GLAC_checker, format_meta_data
from swmf.monitors.OOD import OOD_BetaQuantile
from swmf.monitors.OMS.bam import OMS_BAM_YOLO_logits

In [ ]:
### Reproducibility ###
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
#######################

### ODD Monitoring Setup

In [ ]:
# Define the ODD monitor (GLAC)
odd_monitor = ODD_GLAC_checker()
odd_monitor

### OOD Monitoring Setup

In [ ]:
# Define the OOD monitor (Beta distributions on image props) and train it
ood_monitor = OOD_BetaQuantile()
ood_monitor.fit(trainval_loader, verbose=True)
ood_monitor

### OMS Monitoring Setup

In [ ]:
# Define the OMS monitor and train it
oms_monitor = OMS_BAM_YOLO_logits(yolo_model, density="auto")
oms_monitor.fit(train_loader, verbose=True, keep_only_good=True, iou_thresh=IOU_THRESHOLD, iou_method="CIOU")

# # Calibrate the OMS monitor
# oms_y_true_score_valid = np.where(match_predictions(y_pred_valid, y_true_valid, iou_thresh=IOU_THRESHOLD, iou_method="CIOU")[0] != -1, 1, 0)
# oms_y_pred_score_valid = torch.Tensor([], device="cpu")

# for X, y in tqdm.tqdm(valid_loader, desc="[VALID] Predicting..."):
#     oms_y_pred_score_valid = torch.cat([oms_y_pred_score_valid, torch.Tensor(oms_monitor.score_tensor(X, **ML_KWARGS))])

# oms_y_pred_score_valid = oms_y_pred_score_valid.detach().cpu().numpy()
# oms_thresh_score_valid = np.percentile(oms_y_pred_score_valid[oms_y_true_score_valid == 0], 95)

# print(f"OMS monitor threshold score: {oms_thresh_score_valid}")

<div style="text-align: justify">

According to (He et al., 2025), calibration of the OMS monitor using the FPR95 (FPR@TPR=.95). So we need to compute the TPR of the OMS monitor and set its score threshold to when the TPR is at 95%. We derive 2 possible ways of interpretating this step as provided in (He et al., 2025):

- Compute the TPR of the IMS (in-model-scope) data (False Alarm Probability)

$$ \text{TPR}^{[\text{IMS}]} = \frac{num\_accepted\_IMS}{num\_IMS}$$

- Compute the TPR of the OMS (out-of-model-scope) data (Recall)

$$ \text{TPR}^{[\text{OMS}]} = \frac{num\_rejected\_OMS}{num\_OMS}$$

</div>

In [ ]:
# Compute the ground truths of the OMS monitor (1=should reject (OMS), 0=should accept(IMS))
oms_true_valid = np.where(match_predictions(y_pred_valid, y_true_valid, iou_thresh=IOU_THRESHOLD)[0] != -1, 0, 1)  # 0 = TP/IMS, 1 = FP/OMS

# COmpute the predictions of the OMS monitor on the valid set
oms_pred_valid = torch.Tensor([])
for X, y in tqdm.tqdm(valid_loader, desc="[VALID] Predicting OMS monitoring scores..."):
    oms_pred_valid = torch.cat([oms_pred_valid, torch.Tensor(oms_monitor.score_tensor(X, **ML_KWARGS))])
oms_pred_valid = oms_pred_valid.detach().cpu().numpy()

# Compute the 95 TPR percentile
ii = np.argsort(oms_pred_valid) # sort oms scores
oms_pred_valid_sorted = oms_pred_valid[ii]
oms_true_valid_sorted = oms_true_valid[ii]

In [ ]:
fpr, tpr, thresholds = roc_curve(oms_true_valid, oms_pred_valid)
auc = roc_auc_score(oms_true_valid, oms_pred_valid)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line for random classifier
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Compute TPR_OMS = 95% (i.e., Recall = 0.95)
tpr_percentile = 0.95

omsc = oms_true_valid_sorted.cumsum()
tpr_oms = omsc / omsc[-1]
oms_score_thresh_tpr95 = oms_pred_valid_sorted[np.argmax(tpr_oms >= tpr_percentile)]
oms_score_thresh_tpr95

In [ ]:
# Compute TPR_IMS = 95% (i.e., false alarm proba = 0.05)
fpr_percentile = 0.05

imsc = (1 - oms_true_valid_sorted).cumsum()
tpr_ims = imsc / imsc[-1]
oms_score_thresh_fpr05 = oms_pred_valid_sorted[np.argmax(tpr_ims >= 1 - fpr_percentile)]
oms_score_thresh_fpr05

*For the remainder of the experiment, we will use the score threshold computed at TPR_IMS=95%*

### Evaluation on nominal `test` split

In [ ]:
### Reproducibility ###
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
#######################

In [ ]:
# Create the output arrays to store the results
odd_y_pred_mask = torch.Tensor([], device="cpu")
ood_y_pred_mask = torch.Tensor([], device="cpu")
oms_y_pred_mask = torch.Tensor([], device="cpu")

y_pred = torch.Tensor([], device="cpu")
y_true = torch.Tensor([], device="cpu")
metadata = np.empty((0, 9))

# Run inference and store results
imgs_n = 0
for X, y, m in tqdm.tqdm(test_loader, desc="[TEST] Predicting..."):
    r = yolo_model.predict(X, verbose=False, **ML_KWARGS)
    ## Save YOLO predictions
    y_pred = torch.cat([y_pred, postproc_yolo_outputs(r, imgs_n=imgs_n)])
    y_true = torch.cat([y_true, postproc_yolo_targets(y, imgs_n=imgs_n)])
    imgs_n += len(y)
    ## Save metadata
    metadata = np.concatenate([metadata, np.stack(m, axis=0)], axis=0)
    ## ODD monitoring
    odd_y_pred_mask = torch.cat([odd_y_pred_mask, torch.Tensor(odd_monitor.predict(format_meta_data(m)))])
    ## OOD monitoring
    ood_y_pred_mask = torch.cat([ood_y_pred_mask, torch.Tensor(ood_monitor.predict(X))])
    ## OMS monitoring
    oms_y_pred_mask = torch.cat([oms_y_pred_mask, torch.Tensor(oms_monitor.score_tensor(X, **ML_KWARGS))])

## Derive ML predictions after ODD monitoring
id_odd_img_rejected = torch.arange(odd_y_pred_mask.size(dim=0))[odd_y_pred_mask.to(bool)].to(int)

## Derive ML predictions after OOD monitoring
id_ood_img_rejected = torch.arange(ood_y_pred_mask.size(dim=0))[ood_y_pred_mask.to(bool)].to(int)

# Get the ids of the rejected images (at least one rejected pred)
id_oms_img_rejected = torch.unique(y_pred[:, 0][oms_y_pred_mask >= oms_score_thresh_fpr05])
# Get it into binary array
oms_binary_mask_over_images = np.isin(np.arange(imgs_n), id_oms_img_rejected)

# Gather all monitors decision into a dictionary
monitor_rejects_imgs_idx = {
    'ODD': id_odd_img_rejected,
    'OOD': id_ood_img_rejected,
    'OMS': id_oms_img_rejected,
    'ODD-OOD': torch.unique(torch.cat((id_odd_img_rejected, id_ood_img_rejected), dim=0)),
    'ODD-OMS': torch.unique(torch.cat((id_odd_img_rejected, id_oms_img_rejected), dim=0)),
    'OOD-OMS': torch.unique(torch.cat((id_ood_img_rejected, id_oms_img_rejected), dim=0)),
    'ODD-OOD-OMS': torch.unique(torch.cat((id_odd_img_rejected, id_ood_img_rejected, id_oms_img_rejected), dim=0))
}

monitor_rejects_imgs_msk = {
    'ODD': odd_y_pred_mask,
    'OOD': ood_y_pred_mask,
    'OMS': torch.Tensor(oms_binary_mask_over_images),
    'ODD-OOD': torch.Tensor(np.max([odd_y_pred_mask, ood_y_pred_mask], axis=0)),
    'ODD-OMS': torch.Tensor(np.max([odd_y_pred_mask, oms_binary_mask_over_images], axis=0)),
    'OOD-OMS': torch.Tensor(np.max([ood_y_pred_mask, oms_binary_mask_over_images], axis=0)),
    'ODD-OOD-OMS': torch.Tensor(np.max([odd_y_pred_mask, ood_y_pred_mask, oms_binary_mask_over_images], axis=0)),
}

# Compute y_pred and y_true on decisions
print("Computing accepted y_preds and y_trues...")
y_preds_accepted = {}
y_trues_accepted = {}
for mon_id in monitor_rejects_imgs_msk.keys():
    y_preds_accepted[mon_id] = y_pred[~torch.isin(y_pred[:, 0], monitor_rejects_imgs_idx[mon_id])]
    y_trues_accepted[mon_id] = y_true[~torch.isin(y_true[:, 0], monitor_rejects_imgs_idx[mon_id])]

objdet_metrics_dict = {}
safety_metrics_dict = {}

# Compute metrics
print("Computing metrics...")
for mon_id in monitor_rejects_imgs_msk.keys():
    objdet_metrics_dict[mon_id] = compute_metrics(y_preds_accepted[mon_id], y_trues_accepted[mon_id], iou_thresh=IOU_THRESHOLD, t_conf=optimal_conf_threshold)

objdet_metrics_dict['yolov5-n'] = compute_metrics(y_pred, y_true, iou_thresh=IOU_THRESHOLD, t_conf=optimal_conf_threshold)

# Compute safety metrics
print("Computing safety metrics...")
for mon_id in monitor_rejects_imgs_msk.keys():
    safety_metrics_dict[mon_id] = compute_safety_metrics(y_pred, y_true, monitor_rejects_imgs_msk[mon_id], iou_thresh=IOU_THRESHOLD)

# Save to file
result_path = path_to_results / "metrics" / "test" / f"IOU_{int(IOU_THRESHOLD*100):03d}"
result_path.mkdir(exist_ok=True, parents=True)
with open(result_path / "ml_od_metrics_all.json", "w") as f:
    json.dump(objdet_metrics_dict, f)
with open(result_path / "safety_metrics_all.json", "w") as f:
    json.dump(safety_metrics_dict, f)

print(objdet_metrics_dict)

In [ ]:
# Get the complementarity
ODD_npy = monitor_rejects_imgs_msk['ODD'].detach().cpu().numpy()
OOD_npy = monitor_rejects_imgs_msk['OOD'].detach().cpu().numpy()
OMS_npy = monitor_rejects_imgs_msk['OMS'].detach().cpu().numpy()

print("Total image number:", ODD_npy.shape[0])

# Unique ODD
print("Contribution ODD (cumul):", ODD_npy.astype(int).sum())

# Unique OOD
unique_OOD = np.logical_and(OOD_npy, np.logical_not(ODD_npy))
print("Contribution OOD (cumul):", unique_OOD.sum(), "-- and total:", OOD_npy.astype(int).sum())

# Unique OMS
unique_OMS = np.logical_and(OMS_npy, np.logical_not(np.logical_or(ODD_npy, OOD_npy)))
print("Contribution OMS (cumul):", unique_OMS.sum(), "-- and total:", OMS_npy.astype(int).sum())

print("Total image filtered (1):", ODD_npy.astype(int).sum() + unique_OOD.sum() + unique_OMS.sum())
print("Total image filtered (2):", monitor_rejects_imgs_msk['ODD-OOD-OMS'].detach().cpu().numpy().astype(int).sum())

print(monitor_rejects_imgs_msk['ODD-OOD-OMS'].detach().cpu().numpy().astype(int).sum() / ODD_npy.shape[0])

### Evaluation on corrupted dataset

In [ ]:
### Reproducibility ###
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
#######################

In [ ]:
CORRUPTION_SEVERITIES = [1, 2, 3] 
CORRUPTION_NAMES = [
    'Brightness',
    'Gaussian_Noise',
    'Defocus_Blur',
    'Frosted_Blur',
    'Fog',
]

In [ ]:
objdet_metrics_corrupt_dict = {}
safety_metrics_corrupt_dict = {}

for corruption in CORRUPTION_NAMES:
    objdet_metrics_corrupt_dict[corruption] = {}
    safety_metrics_corrupt_dict[corruption] = {}

    for severity in CORRUPTION_SEVERITIES:
        objdet_metrics_corrupt_dict[corruption][severity] = {}
        safety_metrics_corrupt_dict[corruption][severity] = {}

        print(f"Running for CORRUPTION={corruption}, SEVERITY={severity}")

        # Define the data loader for given corruption and severity
        image_dpath = path_to_lard / "images/test_corrupt" / f"{corruption}_severity_{severity}"
        label_dpath = path_to_lard / "task_detect/labels/test"
        test_corruption_loader = create_dataloader(
            image_dpath=image_dpath,
            label_dpath=label_dpath,
            image_transform=default_img_transform(IMGSZ),
            label_transform=default_lab_transform(IMGSZ),
            yolo_task="detect",
            data_mode="test",
            using_metadata=True,
            batch_size=BATCH,
            shuffle=False,
        )

        # Create the output arrays to store the results
        odd_y_pred_mask_corrupt = torch.Tensor([], device="cpu")
        ood_y_pred_mask_corrupt = torch.Tensor([], device="cpu")
        oms_y_pred_mask_corrupt = torch.Tensor([], device="cpu")

        y_pred_corrupt = torch.Tensor([], device="cpu")
        y_true_corrupt = torch.Tensor([], device="cpu")
        metadata_corrupt = np.empty((0, 9))

        # Run inference and store results
        imgs_n = 0
        for X, y, m in tqdm.tqdm(test_corruption_loader, desc=f"[TEST] Predicting ({corruption}-{severity})..."):
            r = yolo_model.predict(X, verbose=False, **ML_KWARGS)
            ## Save YOLO predictions
            y_pred_corrupt = torch.cat([y_pred_corrupt, postproc_yolo_outputs(r, imgs_n=imgs_n)])
            y_true_corrupt = torch.cat([y_true_corrupt, postproc_yolo_targets(y, imgs_n=imgs_n)])
            imgs_n += len(y)
            ## Save metadata
            metadata_corrupt = np.concatenate([metadata_corrupt, np.stack(m, axis=0)], axis=0)
            ## ODD monitoring
            odd_y_pred_mask_corrupt = torch.cat([odd_y_pred_mask_corrupt, torch.Tensor(odd_monitor.predict(format_meta_data(m)))])
            ## OOD monitoring
            ood_y_pred_mask_corrupt = torch.cat([ood_y_pred_mask_corrupt, torch.Tensor(ood_monitor.predict(X))])
            ## OMS monitoring
            oms_y_pred_mask_corrupt = torch.cat([oms_y_pred_mask_corrupt, torch.Tensor(oms_monitor.score_tensor(X, **ML_KWARGS))])

        # ODD monitoring
        id_odd_img_rejected_corrupt = torch.arange(odd_y_pred_mask_corrupt.size(dim=0))[odd_y_pred_mask_corrupt.to(bool)].to(int)
        # OOD monitoring
        id_ood_img_rejected_corrupt = torch.arange(ood_y_pred_mask_corrupt.size(dim=0))[ood_y_pred_mask_corrupt.to(bool)].to(int)
        # OMS monitor
        id_oms_img_rejected_corrupt = torch.unique(y_pred_corrupt[:, 0][oms_y_pred_mask_corrupt >= oms_score_thresh_fpr05])
        oms_binary_mask_over_images_corrupt = np.isin(np.arange(imgs_n), id_oms_img_rejected_corrupt)

        # Store monitors decisions
        monitor_rejects_imgs_idx_corrupt = {
            'ODD': id_odd_img_rejected_corrupt,
            'OOD': id_ood_img_rejected_corrupt,
            'OMS': id_oms_img_rejected_corrupt,
            'ODD-OOD': torch.unique(torch.cat([id_odd_img_rejected_corrupt, id_ood_img_rejected_corrupt], dim=0)),
            'ODD-OMS': torch.unique(torch.cat([id_odd_img_rejected_corrupt, id_oms_img_rejected_corrupt], dim=0)),
            'OOD-OMS': torch.unique(torch.cat([id_ood_img_rejected_corrupt, id_oms_img_rejected_corrupt], dim=0)),
            'ODD-OOD-OMS': torch.unique(torch.cat([id_odd_img_rejected_corrupt, id_ood_img_rejected_corrupt, id_oms_img_rejected_corrupt], dim=0)),
        }


        monitor_rejects_imgs_msk_corrupt = {
            'ODD': odd_y_pred_mask_corrupt,
            'OOD': ood_y_pred_mask_corrupt,
            'OMS': torch.Tensor(oms_binary_mask_over_images_corrupt),
            'ODD-OOD': torch.Tensor(np.max([odd_y_pred_mask_corrupt, ood_y_pred_mask_corrupt], axis=0)),
            'ODD-OMS': torch.Tensor(np.max([odd_y_pred_mask_corrupt, oms_binary_mask_over_images_corrupt], axis=0)),
            'OOD-OMS': torch.Tensor(np.max([ood_y_pred_mask_corrupt, oms_binary_mask_over_images_corrupt], axis=0)),
            'ODD-OOD-OMS': torch.Tensor(np.max([odd_y_pred_mask_corrupt, ood_y_pred_mask_corrupt, oms_binary_mask_over_images_corrupt], axis=0)),
        }

        # Compute y_pred and y_true on decisions
        print("Computing accepted preds and trues")
        y_preds_accepted_corrupt = {}
        y_trues_accepted_corrupt = {}
        for mon_id in monitor_rejects_imgs_msk_corrupt.keys():
            y_preds_accepted_corrupt[mon_id] = y_pred_corrupt[~torch.isin(y_pred_corrupt[:, 0], monitor_rejects_imgs_idx_corrupt[mon_id])]
            y_trues_accepted_corrupt[mon_id] = y_true_corrupt[~torch.isin(y_true_corrupt[:, 0], monitor_rejects_imgs_idx_corrupt[mon_id])]

        # Compute metrics
        print("Computing metrics")
        for mon_id in monitor_rejects_imgs_msk_corrupt.keys():
            objdet_metrics_corrupt_dict[corruption][severity][mon_id] = compute_metrics(y_preds_accepted_corrupt[mon_id], y_trues_accepted_corrupt[mon_id], iou_thresh=IOU_THRESHOLD, t_conf=optimal_conf_threshold)
        
        objdet_metrics_corrupt_dict[corruption][severity]['yolov5-n'] = compute_metrics(y_pred_corrupt, y_true_corrupt, iou_thresh=IOU_THRESHOLD, t_conf=optimal_conf_threshold)

        # Compute safety metrics
        print("Computing safety metrics")
        for mon_id in monitor_rejects_imgs_msk_corrupt.keys():
            safety_metrics_corrupt_dict[corruption][severity][mon_id] = compute_safety_metrics(y_pred_corrupt, y_true_corrupt, monitor_rejects_imgs_msk_corrupt[mon_id], iou_thresh=IOU_THRESHOLD)
        
        # Save to file
        result_path = path_to_results / "metrics" / "corrupt" / f"IOU_{int(IOU_THRESHOLD*100):03d}" / f"{corruption}_severity_{severity}"
        result_path.mkdir(exist_ok=True, parents=True)
        with open(result_path / "ml_od_metrics.json", "w") as f:
            json.dump(objdet_metrics_corrupt_dict[corruption][severity], f)
        with open(result_path / "safety_metrics.json", "w") as f:
            json.dump(safety_metrics_corrupt_dict[corruption][severity], f)

In [ ]:
result_path = path_to_results / "metrics" / "corrupt" / f"IOU_{int(IOU_THRESHOLD*100):03d}"

with open(result_path / f"all_ml_od_metrics.json", "w") as f:
    json.dump(objdet_metrics_corrupt_dict, f)
with open(result_path / f"all_safety_metrics.json", "w") as f:
    json.dump(safety_metrics_corrupt_dict, f)

## Analysis (table conception)

### Setup

In [43]:
from pathlib import Path
import pandas as pd

PATH_LARD = "data/datasets/lard_512x512_ICPR2026"
PATH_YOLO = "data/models/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/020_epochs/best.pt"

path_to_yolo = Path(PATH_YOLO).resolve()
assert path_to_yolo.exists(), f"ERROR: Specified path does not exist {path_to_yolo}"

path_to_lard = Path(PATH_LARD).resolve()
assert path_to_lard.exists(), f"ERROR: Specified path does not exist {path_to_lard}"

split_fpath = path_to_lard / "split_trainval_per_runway.csv"
split_dname = split_fpath.stem
assert split_fpath.exists(), f"ERROR: Specified path does not exist {split_fpath}"

path_to_results = Path("results").resolve()
path_to_results = path_to_results / PATH_LARD.split('/')[-1] / "detect" / "yolov5n" / (path_to_lard.stem + "_" + split_fpath.stem) / "020_epochs" / split_dname
path_to_results

IOU_THRESHOLD = 0.70  # The IOU threshold to evaluate on

### Analysis of standard metrics (on nominal TEST set)

In [44]:
objdet_metrics_test_path = path_to_results / "metrics" / "test" / f"IOU_{int(IOU_THRESHOLD*100):03d}" / "ml_od_metrics_all.json"

objdet_metrics_test_df = pd.read_json(objdet_metrics_test_path, orient="index")
display(objdet_metrics_test_df)

,ap,f1,p,r,c,tp,fp,fn,n_p,n_t
ODD,0.862529,0.829108,0.872020,0.790222,0.694863,1390,204,369,1594,1759
OOD,0.852733,0.821660,0.868699,0.779453,0.694863,1396,211,395,1607,1791
OMS,0.866390,0.836287,0.876386,0.799697,0.694863,1581,223,396,1804,1977
ODD-OOD,0.857975,0.825119,0.869231,0.785268,0.694863,1130,170,309,1300,1439
ODD-OMS,0.871722,0.841523,0.877800,0.808125,0.694863,1293,180,307,1473,1600
OOD-OMS,0.863472,0.835394,0.877049,0.797516,0.694863,1284,180,326,1464,1610
ODD-OOD-OMS,0.867930,0.838915,0.877398,0.803667,0.694863,1052,147,257,1199,1309
yolov5-n,0.855046,0.822381,0.868712,0.780741,0.694863,1727,261,485,1988,2212


In [45]:
column_to_export = ['p', 'r', 'tp', 'fp', 'fn', 'n_p', 'n_t']

# Export as string to place in LateX file
print(objdet_metrics_test_df.to_latex(
    float_format="%.3f",
    columns=column_to_export,
    column_format="c|"+"c"*len(column_to_export)+"",
    caption="Caption",
    label="tab:standard-metrics-test",
    position="t",
))

\begin{table}[t]
\caption{Caption}
\label{tab:standard-metrics-test}
\begin{tabular}{c|ccccccc}
\toprule
 & p & r & tp & fp & fn & n_p & n_t \\
\midrule
ODD & 0.872 & 0.790 & 1390 & 204 & 369 & 1594 & 1759 \\
OOD & 0.869 & 0.779 & 1396 & 211 & 395 & 1607 & 1791 \\
OMS & 0.876 & 0.800 & 1581 & 223 & 396 & 1804 & 1977 \\
ODD-OOD & 0.869 & 0.785 & 1130 & 170 & 309 & 1300 & 1439 \\
ODD-OMS & 0.878 & 0.808 & 1293 & 180 & 307 & 1473 & 1600 \\
OOD-OMS & 0.877 & 0.798 & 1284 & 180 & 326 & 1464 & 1610 \\
ODD-OOD-OMS & 0.877 & 0.804 & 1052 & 147 & 257 & 1199 & 1309 \\
yolov5-n & 0.869 & 0.781 & 1727 & 261 & 485 & 1988 & 2212 \\
\bottomrule
\end{tabular}
\end{table}



### Analysis of safety metrics (on nominal TEST set)

In [46]:
safety_metrics_test_path = path_to_results / "metrics" / "test" / f"IOU_{int(IOU_THRESHOLD*100):03d}" / "safety_metrics_all.json"

safety_metrics_test_df = pd.read_json(safety_metrics_test_path, orient="index")
display(safety_metrics_test_df)

,SG,RH,AC
ODD,0.046112,0.155515,0.158680
OOD,0.040687,0.160940,0.149638
OMS,0.039783,0.161844,0.066456
ODD-OOD,0.074141,0.127486,0.275316
ODD-OMS,0.074141,0.127486,0.202532
OOD-OMS,0.070524,0.131103,0.201627
ODD-OOD-OMS,0.096745,0.104882,0.311483


In [47]:
column_to_export = ['SG', 'RH', 'AC']

# Export as string to place in LateX file
print(safety_metrics_test_df.to_latex(
    float_format="%.3f",
    columns=column_to_export,
    column_format="c|"+"c"*len(column_to_export)+"",
    caption="Caption",
    label="tab:safety-metrics-test",
    position="t",
))

\begin{table}[t]
\caption{Caption}
\label{tab:safety-metrics-test}
\begin{tabular}{c|ccc}
\toprule
 & SG & RH & AC \\
\midrule
ODD & 0.046 & 0.156 & 0.159 \\
OOD & 0.041 & 0.161 & 0.150 \\
OMS & 0.040 & 0.162 & 0.066 \\
ODD-OOD & 0.074 & 0.127 & 0.275 \\
ODD-OMS & 0.074 & 0.127 & 0.203 \\
OOD-OMS & 0.071 & 0.131 & 0.202 \\
ODD-OOD-OMS & 0.097 & 0.105 & 0.311 \\
\bottomrule
\end{tabular}
\end{table}



### Analysis of standard and safety metrics on corrupt TEST set

In [48]:
objdet_metrics_test_corrupt_path = path_to_results / "metrics" / "corrupt" / f"IOU_{int(IOU_THRESHOLD*100):03d}" / "all_ml_od_metrics.json"

with open(objdet_metrics_test_corrupt_path, 'r') as f:
    dat = json.load(f)

all_metrics = []
for c_k, c_v in dat.items():
    for s_k, s_v in c_v.items():
        for m_k, m_v in s_v.items():
            ret = dict(**m_v)
            ret['Corruption'] = c_k
            ret['Severity'] = s_k
            ret['Monitoring'] = m_k
            all_metrics.append(ret.copy())

df = pd.DataFrame.from_records(all_metrics)
df[df['Monitoring'].isin(('OMS', 'yolov5-n'))]

,ap,f1,p,r,c,tp,fp,fn,n_p,n_t,Corruption,Severity,Monitoring
2,0.792097,0.778691,0.831216,0.732409,0.694894,1374.0,279.0,502.0,1653.0,1876.0,Brightness,1,OMS
7,0.776471,0.758319,0.818993,0.706015,0.694894,1561.0,345.0,650.0,1906.0,2211.0,Brightness,1,yolov5-n
10,0.639061,0.638394,0.725000,0.570272,0.691052,986.0,374.0,743.0,1360.0,1729.0,Brightness,2,OMS
15,0.608012,0.607692,0.701599,0.535957,0.694471,1185.0,504.0,1026.0,1689.0,2211.0,Brightness,2,yolov5-n
18,0.495132,0.496628,0.640737,0.405440,0.694597,626.0,351.0,918.0,977.0,1544.0,Brightness,3,OMS
23,0.450421,0.461710,0.604243,0.373587,0.694597,826.0,541.0,1385.0,1367.0,2211.0,Brightness,3,yolov5-n
26,0.771175,0.728070,0.880303,0.620726,0.694629,1162.0,158.0,710.0,1320.0,1872.0,Gaussian_Noise,1,OMS
31,0.770532,0.716195,0.875245,0.606061,0.694629,1340.0,191.0,871.0,1531.0,2211.0,Gaussian_Noise,1,yolov5-n
34,0.549031,0.398474,0.851449,0.260100,0.694993,470.0,82.0,1337.0,552.0,1807.0,Gaussian_Noise,2,OMS
39,0.588874,0.404417,0.852984,0.265038,0.694993,586.0,101.0,1625.0,687.0,2211.0,Gaussian_Noise,2,yolov5-n


In [49]:
safety_metrics_test_corrupt_path = path_to_results / "metrics" / "corrupt" / f"IOU_{int(IOU_THRESHOLD*100):03d}" / "all_safety_metrics.json"

with open(safety_metrics_test_corrupt_path, 'r') as f:
    dat = json.load(f)

all_metrics = []
for c_k, c_v in dat.items():
    for s_k, s_v in c_v.items():
        for m_k, m_v in s_v.items():
            ret = dict(**m_v)
            ret['Corruption'] = c_k
            ret['Severity'] = s_k
            ret['Monitoring'] = m_k
            all_metrics.append(ret.copy())

safety_df = pd.DataFrame.from_records(all_metrics)
safety_df

,SG,RH,AC,Corruption,Severity,Monitoring
0,0.066486,0.214835,0.138399,Brightness,1,ODD
1,0.238354,0.042967,0.571687,Brightness,1,OOD
2,0.066033,0.215287,0.085482,Brightness,1,OMS
3,0.249661,0.031660,0.603347,Brightness,1,ODD-OOD
4,0.112166,0.169154,0.201719,Brightness,1,ODD-OMS
...,...,...,...,...,...,...
100,0.080054,0.213026,0.090457,Fog,3,OMS
101,0.285844,0.007237,0.675260,Fog,3,ODD-OOD
102,0.127996,0.165084,0.198553,Fog,3,ODD-OMS
103,0.283582,0.009498,0.671642,Fog,3,OOD-OMS


In [51]:
safety_df_OOD = safety_df.loc[safety_df['Monitoring']=='OOD'].drop(columns=['Monitoring']).groupby(by=['Corruption','Severity']).agg('mean')
safety_df_OMS = safety_df.loc[safety_df['Monitoring']=='OMS'].drop(columns=['Monitoring']).groupby(by=['Corruption','Severity']).agg('mean')
safety_df_all = safety_df.loc[safety_df['Monitoring']=='OOD-OMS'].drop(columns=['Monitoring']).groupby(by=['Corruption','Severity']).agg('mean')

safety_df_final = pd.concat((safety_df_OOD, safety_df_OMS, safety_df_all), axis=1)
safety_df_final.columns = pd.MultiIndex.from_product([['OOD','OMS','OOD-OMS'], safety_df_OOD.columns])
safety_df_final

OOD                           OMS            \
                               SG        RH        AC        SG        RH   
Corruption     Severity                                                     
Brightness     1         0.238354  0.042967  0.571687  0.066033  0.215287   
               2         0.446404  0.005427  0.537313  0.128449  0.323383   
               3         0.604704  0.000000  0.395296  0.203528  0.401176   
Defocus_Blur   1         0.073270  0.173225  0.266848  0.054726  0.191768   
               2         0.421076  0.000000  0.578924  0.094075  0.327001   
               3         0.616011  0.000000  0.383989  0.130258  0.485753   
Fog            1         0.088648  0.152872  0.310719  0.049751  0.191768   
               2         0.198553  0.062415  0.587065  0.056988  0.203980   
               3         0.280868  0.012212  0.666214  0.080054  0.213026   
Frosted_Blur   1         0.052465  0.219810  0.143826  0.049299  0.222976   
               2         0.060154  0.327454  0.133876  0.061963  0.325645   
               3         0.283582  0.123926  0.433288  0.083220  0.324288   
Gaussian_Noise 1         0.319313  0.000000  0.678878  0.038896  0.280416   
               2         0.599729  0.000000  0.400271  0.037087  0.562641   
               3         0.815920  0.000000  0.184080  0.046585  0.769335   

                                    OOD-OMS                      
                               AC        SG        RH        AC  
Corruption     Severity                                          
Brightness     1         0.085482  0.247852  0.033469  0.589326  
               2         0.089552  0.446857  0.004975  0.538670  
               3         0.098146  0.604704  0.000000  0.395296  
Defocus_Blur   1         0.081863  0.105382  0.141113  0.317051  
               2         0.115785  0.421076  0.000000  0.578924  
               3         0.153777  0.616011  0.000000  0.383989  
Fog            1         0.076436  0.115332  0.126187  0.351425  
               2         0.086386  0.210764  0.050204  0.601990  
               3         0.090457  0.283582  0.009498  0.671642  
Frosted_Blur   1         0.092266  0.088648  0.183627  0.218453  
               2         0.107644  0.112166  0.275441  0.214383  
               3         0.104025  0.306196  0.101312  0.459521  
Gaussian_Noise 1         0.114428  0.319313  0.000000  0.679331  
               2         0.145635  0.599729  0.000000  0.400271  
               3         0.084577  0.815920  0.000000  0.184080

In [52]:
# Export as string to place in LateX file
print(safety_df_final.to_latex(
    float_format="%.3f",
    caption="Caption",
    label="tab:safety-metrics-test-corrupt",
    position="t",
))

\begin{table}[t]
\caption{Caption}
\label{tab:safety-metrics-test-corrupt}
\begin{tabular}{llrrrrrrrrr}
\toprule
 &  & \multicolumn{3}{r}{OOD} & \multicolumn{3}{r}{OMS} & \multicolumn{3}{r}{OOD-OMS} \\
 &  & SG & RH & AC & SG & RH & AC & SG & RH & AC \\
Corruption & Severity &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{3}{*}{Brightness} & 1 & 0.238 & 0.043 & 0.572 & 0.066 & 0.215 & 0.085 & 0.248 & 0.033 & 0.589 \\
 & 2 & 0.446 & 0.005 & 0.537 & 0.128 & 0.323 & 0.090 & 0.447 & 0.005 & 0.539 \\
 & 3 & 0.605 & 0.000 & 0.395 & 0.204 & 0.401 & 0.098 & 0.605 & 0.000 & 0.395 \\
\cline{1-11}
\multirow[t]{3}{*}{Defocus_Blur} & 1 & 0.073 & 0.173 & 0.267 & 0.055 & 0.192 & 0.082 & 0.105 & 0.141 & 0.317 \\
 & 2 & 0.421 & 0.000 & 0.579 & 0.094 & 0.327 & 0.116 & 0.421 & 0.000 & 0.579 \\
 & 3 & 0.616 & 0.000 & 0.384 & 0.130 & 0.486 & 0.154 & 0.616 & 0.000 & 0.384 \\
\cline{1-11}
\multirow[t]{3}{*}{Fog} & 1 & 0.089 & 0.153 & 0.311 & 0.050 & 0.192 & 0.076 & 0.115 & 0.126 & 0.351 \\
 & 2 & 0.199 &